# Overview

In [ ]:
chunk_start

In [ ]:
import datetime
from oceanval import analysis_plots as ap

# what analyze() saved, beside the report
results_dir = "../../results"
info = ap.load(results_dir)


def years_text(analysis):
    if analysis["start"] == analysis["end"]:
        return str(analysis["start"])
    return f"{analysis['start']}-{analysis['end']}"

In [ ]:
created = datetime.datetime.fromisoformat(info["created"])
count = len(info["variables"])
md(
    f"This report analyses the simulation in `{info['sim_dir']}`, on its own, with no observations. "
    f"It covers {count} variable{'s' if count != 1 else ''}, and was made with **oceanval** "
    f"{info['oceanval']} on {created.day} {created.strftime('%B %Y')}. "
    "Each variable has a page of its own, listed on the left."
)

In [ ]:
rows = []
for entry in info["variables"]:
    rows.append(
        {
            "Variable": entry["name"],
            "Model variable": entry["model_variable"].replace("+", " + "),
            "Unit": ap.unit_html(entry["units"]),
            "Files": f"{entry['pattern']} ({entry['files']})",
            "Output": entry["time_res_text"].capitalize(),
            "Dates": f"{entry['first']} to {entry['last']}",
        }
    )
pd.DataFrame(rows).style.hide(axis="index")

In [ ]:
md(f"**Table {i_table}:** The variables analysed, the model variables holding them, and the output they are read from: the files, named with their dates left out, and how many there are.")
i_table += 1

In [ ]:
rows = []
for entry in info["variables"]:
    clim = entry.get("climatology")
    phen = entry.get("phenology")
    long_term = entry.get("long_term")
    seasonal = entry.get("seasonal")
    row = {
        "Variable": entry["name"],
        "Climatology": (
            f"{years_text(clim)}: " + ", ".join(layer["title"].lower() for layer in clim["layers"].values())
            if clim else "None"
        ),
        "Phenology": f"{years_text(phen)}: surface" if phen else "None",
    }
    if any(x.get("long_term") for x in info["variables"]):
        row["Long-term series"] = (
            f"{years_text(long_term)}: " + ", ".join(layer["title"].lower() for layer in long_term["layers"].values())
            if long_term else "None"
        )
    if any(x.get("seasonal") for x in info["variables"]):
        row["Seasonal series"] = (
            f"{years_text(seasonal)}: " + ", ".join(layer["title"].lower() for layer in seasonal["layers"].values())
            if seasonal else "None"
        )
    if any(x.get("transects") for x in info["variables"]):
        transects = entry.get("transects")
        row["Transects"] = (
            f"{years_text(transects)}: {len(transects['lines'])}" if transects else "None"
        )
    rows.append(row)
pd.DataFrame(rows).style.hide(axis="index")

In [ ]:
md(f"**Table {i_table}:** What was worked out for each variable, and over which years.")
i_table += 1

In [ ]:
notes = []
for entry in info["variables"]:
    for kind in ("climatology", "phenology", "long_term", "seasonal", "transects"):
        analysis = entry.get(kind)
        if analysis and analysis["dates"]["issues"]:
            notes.append(f"- **{entry['name']}**, {kind.replace('_', '-')} ({years_text(analysis)}): " + " ".join(analysis["dates"]["issues"]))
md("## Gaps in the output")
if notes:
    md("The output is missing times in the years some of the variables were analysed for. Averages are taken over the times there are.\n\n" + "\n".join(notes))
else:
    md("No times are missing from the output in the years each variable was analysed for.")

In [ ]:
md("## Methods")
methods = [
    "**Climatology.** For each variable, the mean of each month of each year is taken first, so that a month of daily output counts as much as a month of monthly output. The climatology is the mean of each month of the year over the years chosen, and the annual mean is the mean of those monthly means. "
    "At the surface, the first value from the top of the water column that is not missing is taken, as in OceanVal's matchups. "
    "At the bottom, the deepest value that is not missing is taken (nctoolkit's `bottom` method with `choice=\"value\"`), so in each grid cell it is that of the deepest level with water, whatever level that is. "
    "The water column's vertical mean weights each layer by its thickness; its depth integral multiplies each layer by its thickness and sums them, so a concentration per cubic metre becomes an amount per square metre.",
    "**Phenology.** The day of the seasonal peak is found in each grid cell from the variable's daily climatology at the surface: its mean on each day of the year over the years chosen. The peak is the day with the highest mean, or the first of them if there are more than one, and days are counted from 1 January in the model's own calendar.",
]
if any(entry.get("long_term") for entry in info["variables"]):
    methods.append(
        "**Long-term time series.** The mean of each month of each year is taken first, as for the climatology, and a year's value is the mean of its months' means, "
        "so that a year with output missing is the mean of the months there are: it is shown as an open circle, with the months it is the mean of in its data file. "
        "The domain and region values are area-weighted spatial means of the monthly fields."
    )
if any(entry.get("seasonal") for entry in info["variables"]):
    methods.append(
        "**Seasonal time series.** The mean of each month of each year, over the domain, over each region and at each station: a month with no output is a gap. "
        "The mean seasonal cycle is the mean of each month of the year over the years of the series."
    )
if any(entry.get("transects") for entry in info["variables"]):
    methods.append(
        "**Transects.** Each transect is a straight line that runs north-south or east-west, with the options chosen for it and the years of each. The mean of each month of each year is taken first, as for the climatology, and extracted along the line with nctoolkit's `to_transect` (bilinear interpolation onto evenly spaced points, about one for each grid cell the line crosses), at the surface (the first value from the top that is not missing) or, for the *vertical climatology* option, through the water column. "
        "The *surface climatology* is the mean of each month of the year over the years chosen, and the *vertical climatology* section is the annual mean, the mean of those monthly means, at each depth. "
        "An *absolute trend* is the slope of a straight line fitted by least squares to the annual means at the surface, per year, where a year's mean is the mean of its months' means and a year with months missing is left out; it needs at least three such years. "
        "The *absolute change* is the annual mean of the last year chosen minus that of the first, and the *percentage change* is that as a percentage of the absolute value of the first year's mean, so that an increase is positive. "
        "The annual means are worked out with nctoolkit and, if a rolling mean window of more than one year was chosen for a trend, smoothed by nctoolkit's `rolling_mean` before the trend is worked out, so that a change is then the mean of the last years in the window minus the mean of the first."
    )
if info.get("subregions"):
    which = {"nwes": "the northwest European Shelf regions", "global": "the global regions"}.get(info["subregions"], f"the regions in `{info['subregions']}`")
    methods.append(
        f"**Regions.** The regional summaries use {which}, as OceanVal's validation does: each region's map is put on the model's grid by nearest neighbour, the model's field is multiplied by it, and its area-weighted spatial mean taken. "
        "A region with no cells in the model's domain is left out."
    )
if info.get("lon_lim"):
    methods.append(
        f"**Spatial subset.** The domain and region means of the time series are of the part of the model's domain in longitudes {info['lon_lim'][0]:g} to {info['lon_lim'][1]:g} and latitudes {info['lat_lim'][0]:g} to {info['lat_lim'][1]:g}. The stations are not cut by it."
    )
if info.get("stations"):
    methods.append(
        "**Stations.** The series at a station are those of the cell of the model's grid, with values, that is nearest to it, by distance over the Earth's surface. The cells used are in the tables on each variable's page."
    )
if info.get("as_missing") is not None:
    missing = info["as_missing"]
    what = f"{missing[0]:g}" if not isinstance(missing, list) else (f"{missing[0]:g}" if missing[0] == missing[1] else f"{missing[0]:g} to {missing[1]:g}")
    methods.append(f"Values of {what} in the model's output were treated as missing, as were those of the cell thickness.")
if info.get("thickness"):
    if info["thickness"] == "z_level":
        methods.append("The water column is on fixed depth levels (z-levels), whose thicknesses are taken from the levels themselves.")
    else:
        methods.append(f"The cell thickness of the water column is `{info['thickness']}`, at its first time if it changes over time.")
if info.get("time_shift"):
    days = info["time_shift"]
    way = "back" if days < 0 else "forward"
    methods.append(f"The model's times were moved {way} {abs(days)} day{'s' if abs(days) != 1 else ''} before they were used, as they mark the end of the period each mean covers.")
md("\n\n".join(methods))

In [ ]:
chunk_end